# 09b - Feature engineering: regression GBM + stacked feature test

First of three tuning stages: feature set here, individual base-learner hyperparameters next (`09c`), ensemble weights last (`09d`).
Builds and lightly tunes a `lateness` regression GBM, then tests whether stacking its predicted lateness in as an extra classifier feature helps XGBoost/RF. Tested on `GBM_FEATURES`, which includes additional feats `stops_from_cc` and `any_service_exception`

## Setup

In [ ]:
import time as _time

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    HistGradientBoostingRegressor, RandomForestClassifier,
)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit, KFold
from sklearn.metrics import (
    mean_absolute_error, root_mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, brier_score_loss,
)

import xgboost as xgb

from utils import (
    load_model_split, prep_gbm_matrices, GBM_FEATURES, GBM_CAT_FEATURES,
)

BASEPATH = "../data"

# temporary progress instrumentation for long-running cells below # Claude
PROGRESS_LOG = "/tmp/09b_progress.log"

def log_progress(msg):
    ts = _time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(PROGRESS_LOG, "a") as f:
        f.write(line + "\n")

log_progress("09b_tuning.ipynb started")

train_df, test_df = load_model_split(BASEPATH)
y_train_otp = train_df["is_otp"]
y_test_otp = test_df["is_otp"]
y_train_log = np.log1p(train_df["lateness"])
print(f"train: {train_df.shape}, test: {test_df.shape}")

[08:47:25] 09b_tuning.ipynb started


train: (1357563, 107), test: (126332, 107)


## 1. Regression GBM: baseline + lightly tuned

Same `log1p(lateness)` framing as the classifier, `GBM_FEATURES`/`GBM_CAT_FEATURES` (including `stops_from_cc`/`any_service_exception`). Modest `RandomizedSearchCV` budget (15 iters x 3-fold `TimeSeriesSplit`) - a quick check to see if I can get any extra gains from tuning before adding the prediction as a stacked feature below

In [2]:
X_train_gbm, X_test_gbm = prep_gbm_matrices(train_df, test_df)

_t0 = _time.time()
reg_baseline = HistGradientBoostingRegressor(
    categorical_features = "from_dtype", random_state = 42
)
reg_baseline.fit(X_train_gbm, y_train_log)

pred_baseline = np.expm1(reg_baseline.predict(X_test_gbm))
print(
    f"Baseline regression GBM: "
    f"MAE={mean_absolute_error(test_df['lateness'], pred_baseline):.3f}, "
    f"RMSE={root_mean_squared_error(test_df['lateness'], pred_baseline):.3f}, "
    f"R2={r2_score(test_df['lateness'], pred_baseline):.3f}"
)

Baseline regression GBM: MAE=3.418, RMSE=8.677, R2=0.248


In [3]:
# TimeSeriesSplit needs chronological order
train_df_sorted = (
    train_df.sort_values("service_date").reset_index(drop = True)
)
X_train_gbm_sorted, _ = prep_gbm_matrices(train_df_sorted, test_df)
y_train_log_sorted = np.log1p(train_df_sorted["lateness"])

param_distributions = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "max_iter": [100, 200, 300, 500],
    "max_leaf_nodes": [15, 31, 63, 127],
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_leaf": [10, 20, 50, 100],
    "l2_regularization": [0, 0.1, 1, 10],
}

_t0 = _time.time()
log_progress(
    "Running RandomizedSearchCV for the regression GBM "
    "(15 iters x 3-fold TimeSeriesSplit)..."
)
search = RandomizedSearchCV(
    HistGradientBoostingRegressor(
        categorical_features = "from_dtype", random_state = 42
    ),
    param_distributions = param_distributions,
    n_iter = 15,
    scoring = "neg_root_mean_squared_error",
    cv = TimeSeriesSplit(n_splits = 3),
    random_state = 42,
    n_jobs = -1,
)
search.fit(X_train_gbm_sorted, y_train_log_sorted)

TUNED_REG_PARAMS = search.best_params_
print(f"Best CV RMSE (log1p-space): {-search.best_score_:.4f}")
print(f"Best params: {TUNED_REG_PARAMS}")

reg_tuned = search.best_estimator_
pred_tuned = np.expm1(reg_tuned.predict(X_test_gbm))
print(
    f"Tuned regression GBM:    "
    f"MAE={mean_absolute_error(test_df['lateness'], pred_tuned):.3f}, "
    f"RMSE={root_mean_squared_error(test_df['lateness'], pred_tuned):.3f}, "
    f"R2={r2_score(test_df['lateness'], pred_tuned):.3f}"
)

[08:47:40] Running RandomizedSearchCV for the regression GBM (15 iters x 3-fold TimeSeriesSplit)...


/Users/stkath/MSE Data Science/DATS_5990_departure_prediction/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best CV RMSE (log1p-space): 0.7693
Best params: {'min_samples_leaf': 10, 'max_leaf_nodes': 127, 'max_iter': 500, 'max_depth': 7, 'learning_rate': 0.03, 'l2_regularization': 1}


Tuned regression GBM:    MAE=3.371, RMSE=8.631, R2=0.256


## 2. Build the stacked lateness feature (out of fold)

The tuned regression GBM's predicted `lateness` becomes an input to the classifier, alongside `GBM_FEATURES`. If we fit this regression on the full training set it'd leak too much info to the binary classifier. Train-side is built w/ k = 5-fold to avoid, shuffling instead of time ordering. Every feature col is leak-safe by design so no issue there. Test side simply predicts directly from reg GBM fit on the full training set.

In [4]:
_t0 = _time.time()
log_progress(
    "Building out-of-fold stacked lateness feature (5-fold KFold)..."
)
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)
oof_pred_lateness = np.zeros(len(train_df))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train_gbm), start = 1):
    _tfold = _time.time()
    fold_reg = HistGradientBoostingRegressor(
        categorical_features = "from_dtype", random_state = 42,
        **TUNED_REG_PARAMS
    )
    fold_reg.fit(X_train_gbm.iloc[tr_idx], y_train_log.iloc[tr_idx])
    oof_pred_lateness[val_idx] = np.expm1(
        fold_reg.predict(X_train_gbm.iloc[val_idx])
    )
    log_progress(f"  fold {fold}/5 done ({_time.time() - _tfold:.1f}s)")

train_df["pred_lateness_gbm"] = oof_pred_lateness

reg_full = HistGradientBoostingRegressor(
    categorical_features = "from_dtype", random_state = 42,
    **TUNED_REG_PARAMS
)
reg_full.fit(X_train_gbm, y_train_log)
test_df["pred_lateness_gbm"] = np.expm1(reg_full.predict(X_test_gbm))
log_progress(f"Stacked feature built ({_time.time() - _t0:.1f}s total)")

[08:56:36] Building out-of-fold stacked lateness feature (5-fold KFold)...


[08:57:24]   fold 1/5 done (48.0s)


[08:58:10]   fold 2/5 done (45.3s)


[08:58:57]   fold 3/5 done (46.7s)


[08:59:50]   fold 4/5 done (53.8s)


[09:00:44]   fold 5/5 done (53.8s)


[09:01:42] Stacked feature built (306.0s total)


## 3. Does the stacked feature help XGBoost/RF

Tested on the two base learners used in the rest of the pipeline, same configs as `09`/`09c`. The baseline
(no stacked feature) is `09_prediction.ipynb`'s own GBM/RF fit -- identical features, hyperparameters,
and train data -- so its cached predictions are loaded directly rather than refitting the same models
here. Only the `+ stacked lateness feature` variant is actually fit in this notebook.

In [ ]:
def evaluate(name, y_true, p_otp, threshold = 0.5):
    y_late = 1 - y_true
    p_late = 1 - p_otp
    pred = (p_otp >= threshold).astype(int)
    return {
        "model": name,
        "accuracy": (pred == y_true).mean(),
        "roc_auc": roc_auc_score(y_true, p_otp),
        "pr_auc_late": average_precision_score(y_late, p_late),
        "brier": brier_score_loss(y_true, p_otp),
    }


def fit_and_eval(name, features, cat_features = GBM_CAT_FEATURES):
    """Fit XGBoost + RF (09/09c configs) on `features`, evaluate on test."""
    X_train, X_test = prep_gbm_matrices(
        train_df, test_df, features = features, cat_features = cat_features
    )

    _t0 = _time.time()
    log_progress(f"[{name}] fitting XGBoost...")
    xgb_model = xgb.XGBClassifier(
        random_state = 42, enable_categorical = True, tree_method = "hist"
    )
    xgb_model.fit(X_train, y_train_otp)
    log_progress(f"[{name}] XGBoost fit complete ({_time.time() - _t0:.1f}s)")
    p_xgb = xgb_model.predict_proba(X_test)[:, 1]

    X_train_rf = pd.get_dummies(train_df[features], columns = cat_features)
    X_test_rf = pd.get_dummies(test_df[features], columns = cat_features)
    X_test_rf = X_test_rf.reindex(columns = X_train_rf.columns, fill_value = 0)

    _t0 = _time.time()
    log_progress(f"[{name}] fitting random forest...")
    rf_model = RandomForestClassifier(
        n_estimators = 200, max_depth = 20, min_samples_leaf = 20,
        n_jobs = -1, random_state = 42,
    )
    rf_model.fit(X_train_rf, y_train_otp)
    log_progress(f"[{name}] RF fit complete ({_time.time() - _t0:.1f}s)")
    p_rf = rf_model.predict_proba(X_test_rf)[:, 1]

    return pd.DataFrame([
        evaluate(f"{name}: XGBoost", y_test_otp, p_xgb),
        evaluate(f"{name}: random forest", y_test_otp, p_rf),
    ]).set_index("model")


# baseline is 09_prediction.ipynb's own GBM/RF fit (identical features,
# hyperparams, train data) -- load its cached predictions instead of
# refitting the same models here
baseline_preds = pd.read_parquet(
    f"{BASEPATH}/9_baseline_predictions_test.parquet"
)
baseline_results = pd.DataFrame([
    evaluate(
        "baseline (no stacked feature): XGBoost",
        baseline_preds["is_otp"], baseline_preds["p_gbm"],
    ),
    evaluate(
        "baseline (no stacked feature): random forest",
        baseline_preds["is_otp"], baseline_preds["p_rf"],
    ),
]).set_index("model")
print(baseline_results)

                                              accuracy   roc_auc  pr_auc_late  \
model                                                                           
baseline (no stacked feature): XGBoost        0.832394  0.807156     0.613955   
baseline (no stacked feature): random forest  0.828792  0.793102     0.588565   

                                                 brier  
model                                                   
baseline (no stacked feature): XGBoost        0.123637  
baseline (no stacked feature): random forest  0.128936  


In [ ]:
STACKED_FEATURES = GBM_FEATURES + ["pred_lateness_gbm"]
stacked_results = fit_and_eval("+ stacked lateness feature", STACKED_FEATURES)
print(pd.concat([baseline_results, stacked_results]))
log_progress("09b_tuning.ipynb complete")